# 게이트가 있는 프롬프트 체이닝

첫 출력이 다음 호출에 들어가기 전에 필요한 항목을 갖췄는지 검사합니다.

## 실행 환경과 비용 상한

최대 3회, 호출당 출력 260토큰으로 제한합니다. 계획 비용은 0.0081달러 미만입니다.

## 코드를 읽기 전에

이 장은 프레임워크가 패턴을 대신 구현하기 전의 흐름을 보기 위해 OpenAI Python SDK를 직접 사용합니다. `OpenAI(...)`는 동기 클라이언트를 만들고, `client.responses.create(...)`를 호출하는 순간 실제 네트워크 요청이 발생합니다. 반환된 응답 객체의 `output_text`는 모델이 만든 텍스트이고, `usage`는 성공 응답에 기록된 입력·출력 토큰입니다. `call_json()`은 JSON 텍스트를 파이썬 사전으로 바꾸며, JSON 객체라는 형식이 업무 필드의 완전성까지 보장하지는 않습니다.

`outline_gate(data)`는 사전을 입력받아 통과 여부와 누락 이유 목록을 반환합니다. Before에서는 첫 호출 뒤 게이트가 실패하므로 두 번째 모델 호출이 일어나지 않습니다. After에서는 검사한 기획만 문자열로 직렬화해 다음 호출에 전달합니다.

In [1]:
# 이 장은 OpenAI SDK를 직접 배우므로 두 패키지만 고정합니다.
%pip install -q openai==2.29.0 python-dotenv==1.2.3


Note: you may need to restart the kernel to use updated packages.


### 다음 셀을 읽는 방법

`load_dotenv(find_dotenv(usecwd=True))`는 현재 폴더에서 위로 올라가며 `.env`를 찾아 환경 변수에 넣습니다. `OpenAI(timeout=45.0, max_retries=0)`는 동기 클라이언트를 만들며 아직 모델을 호출하지 않습니다. `calls`는 이후 성공 응답의 지연과 토큰 수를 모을 빈 목록입니다.


In [2]:
# 동기 API 호출과 사용량 측정에 필요한 도구를 준비합니다.
import json
import os
import time
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "프로젝트 루트 .env에 OPENAI_API_KEY가 필요합니다."
MODEL = "gpt-5.4-mini"
client = OpenAI(timeout=45.0, max_retries=0)
calls: list[dict[str, float | int | str]] = []
print("모델:", MODEL, "| API 키 존재: 예")


모델: gpt-5.4-mini | API 키 존재: 예


### 다음 셀을 읽는 방법

`call_json(instructions, prompt, limit)`은 지시문·사용자 입력·최대 출력 토큰을 받습니다. 함수 안의 `client.responses.create()`에서 실제 요청이 발생하며, 응답의 JSON 텍스트를 파이썬 사전으로 바꾸어 반환합니다. `calls`에는 성공 응답의 지연과 입력·출력 토큰만 추가됩니다.


In [3]:
# 동기 Responses API의 JSON 응답과 사용량을 기록합니다.
def call_json(instructions: str, prompt: str, limit: int = 240) -> dict:
    started = time.perf_counter()
    response = client.responses.create(
        model=MODEL,
        instructions=instructions + " 반드시 유효한 JSON 객체만 반환하세요.",
        input="JSON 요청\n" + prompt,
        max_output_tokens=limit,
        reasoning={"effort": "none"},
        text={"format": {"type": "json_object"}},
    )
    elapsed = time.perf_counter() - started
    usage = response.usage
    calls.append({"latency_s": elapsed, "input": usage.input_tokens,
                  "output": usage.output_tokens})
    return json.loads(response.output_text)


### 다음 셀을 읽는 방법

`call_answer()`는 `answer` 문자열 하나만 허용하는 JSON Schema를 요청에 넣습니다. SDK 응답을 사전으로 바꾼 뒤 값이 문자열이며 비어 있지 않은지 다시 검사하고 사전을 반환합니다. 스키마 통과는 답의 사실 정확성까지 보장하지 않습니다.


In [4]:
# 동기 응답의 answer 형식을 엄격하게 검사합니다.
def call_answer(instructions: str, prompt: str, limit: int = 400) -> dict:
    started = time.perf_counter()
    response = client.responses.create(
        model=MODEL, instructions=instructions, input=prompt,
        max_output_tokens=limit, reasoning={"effort": "none"},
        text={"format": {"type": "json_schema", "name": "answer_payload",
              "schema": {"type": "object", "properties": {"answer": {"type": "string"}},
              "required": ["answer"], "additionalProperties": False}, "strict": True}},
    )
    calls.append({"latency_s": time.perf_counter() - started,
                  "input": response.usage.input_tokens, "output": response.usage.output_tokens})
    data = json.loads(response.output_text)
    if not isinstance(data.get("answer"), str) or not data["answer"].strip():
        raise ValueError("응답의 answer가 비어 있습니다.")
    return data


### 다음 셀을 읽는 방법

`cost_usd(records)`는 앞에서 모은 성공 응답의 입력·출력 토큰을 합산해 이 실습에 고정한 단가를 적용합니다. `score_of(data)`는 평가 점수가 불리언이 아닌 숫자이며 0~100 범위인지 확인한 뒤 정수로 반환합니다. 두 함수 모두 모델을 호출하지 않습니다.


In [5]:
# 반환된 토큰 사용량으로 할인 없는 비용을 추정합니다.
def cost_usd(records: list[dict]) -> float:
    inputs = sum(int(row["input"]) for row in records)
    outputs = sum(int(row["output"]) for row in records)
    return inputs * 0.75 / 1_000_000 + outputs * 4.50 / 1_000_000
def score_of(data: dict) -> int:
    value = data.get("score")
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise ValueError("평가 응답의 score는 숫자여야 합니다.")
    if not 0 <= value <= 100:
        raise ValueError("평가 응답의 score는 0~100이어야 합니다.")
    return int(value)


## 기본 실행: 게이트의 통과 조건 확인

In [6]:
# 완전한 예시로 체이닝 게이트의 통과 조건을 확인합니다.
def outline_gate(data: dict) -> tuple[bool, list[str]]:
    required = {"title", "audience", "steps"}
    missing = sorted(required - data.keys())
    if not isinstance(data.get("steps"), list) or len(data.get("steps", [])) < 3:
        missing.append("steps(3개 이상)")
    return not missing, missing
sample = {"title": "비밀번호 재설정", "audience": "로그인 사용자",
          "steps": ["이메일 입력", "코드 확인", "새 비밀번호 저장"]}
print("완전한 중간 결과:", outline_gate(sample))
assert outline_gate(sample) == (True, [])


완전한 중간 결과: (True, [])


## Before: 불완전한 중간 결과에서 중단

In [7]:
# 필드가 빠진 실제 중간 출력에서 후속 호출을 중단합니다.
before = call_json("한글 안내문 기획자입니다.",
                   "비밀번호 재설정 안내의 title만 JSON으로 작성하세요.", 120)
passed, reasons = outline_gate(before)
print("중간 출력:", before)
print("게이트 통과:", passed, "| 부족한 항목:", reasons)
print("다음 호출 수:", 0 if not passed else 1)
assert not passed and reasons


중간 출력: {'title': '비밀번호 재설정 안내'}
게이트 통과: False | 부족한 항목: ['audience', 'steps', 'steps(3개 이상)']
다음 호출 수: 0


## After: 검사한 출력만 다음 프롬프트에 전달

In [8]:
# 검사에 통과한 기획만 최종 안내문 호출에 전달합니다.
outline = call_json(
    "고객 지원 안내문 기획자입니다.",
    "비밀번호 재설정 안내를 title, audience, steps 3개인 JSON으로 작성하세요.", 180)
passed, reasons = outline_gate(outline)
if not passed:
    final = {"status": "stopped", "reasons": reasons}
else:
    final = call_answer(
        "초보 사용자를 위한 고객 지원 작성자입니다.",
        f"검사된 기획을 4문장 안내문으로 작성하세요. JSON answer로 반환: {outline}", 260)
print("게이트 통과:", passed)
print("최종 결과:", final)
print("누적 성공 응답:", len(calls), "| 사용량 기준 추정 비용($):", round(cost_usd(calls), 6))
assert passed and final["answer"].strip()


게이트 통과: True
최종 결과: {'answer': "비밀번호 재설정 안내입니다. 로그인 화면에서 '비밀번호 찾기' 또는 '비밀번호 재설정'을 선택해 주세요. 등록한 이메일 또는 휴대폰 번호를 입력한 뒤 인증 절차를 완료해 주세요. 새 비밀번호를 설정하고 저장한 후 다시 로그인하시면 됩니다."}
누적 성공 응답: 3 | 사용량 기준 추정 비용($): 0.001007


## 다른 상황에 적용하기

번역 뒤 법률 용어 검사를 거쳐 요약하는 흐름에서도 중간 게이트를 둘 수 있습니다. 필수 용어 누락처럼 코드로 확인할 조건은 프로그램이 검사하고, 문체 자연스러움처럼 판단 기준이 필요한 조건은 별도 평가가 필요합니다. 게이트는 앞 단계의 오류가 다음 단계로 전파되는 것을 줄이지만, 통과한 내용의 사실 정확성을 자동으로 보장하지는 않습니다.